# 📖 Notebook 1: Click Event Ingestion

In this notebook we'll build the **first stage** of an Ad Click Aggregator: capturing click events from users and flowing them through **Apache Kafka** into **PostgreSQL**.

## Why Kafka?

Imagine 10 000 users clicking ads every second. If every click went straight to the database, the database would choke. **Kafka** acts as a durable buffer (a "shock absorber") between the flood of incoming clicks and the slower database writes.

```
User clicks ad
      │
      ▼
┌──────────────┐      ┌────────────┐      ┌─────────────────┐
│ Click        │─────▶│   Kafka    │─────▶│  Consumer        │
│ Producer     │      │  (buffer)  │      │  (writes to DB) │
└──────────────┘      └────────────┘      └────────┬────────┘
                                                   │
                                                   ▼
                                          ┌─────────────────┐
                                          │   PostgreSQL    │
                                          │  (click_events) │
                                          └─────────────────┘
```

## Learning Objectives

By the end of this notebook you will understand:
- How Kafka producers send messages (click events) to a topic
- How Kafka consumers read messages from a topic
- How to store raw click events in PostgreSQL
- Why we partition by `ad_id` and what that buys us

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/ad-click-aggregator
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `adclick_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`
- **Kafka UI**: http://localhost:8081  
  See topics, partitions, and messages flowing through the click stream

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import json
import time
import uuid
import random
from datetime import datetime, timezone
from confluent_kafka import Producer, Consumer, KafkaError
from confluent_kafka.admin import AdminClient, NewTopic

# ── Determinism ─────────────────────────────────────────────
# Every simulation in this notebook is seeded, so the numbers you read in the
# prose are the numbers you get on your machine. RUN_ID tags the events this
# run produces: the Kafka topic outlives the notebook, so without the tag a
# second run would "consume" the first run's events and quietly describe them.
random.seed(42)
RUN_ID = uuid.uuid4().hex[:8]
print(f"🔖 run id: {RUN_ID}")

# ── Connection settings ─────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "adclick_demo",
    "user": "demo",
    "password": "demo"
}

KAFKA_BROKER = "localhost:9094"  # external listener

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# ── Verify connections ──────────────────────────────────────
try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

try:
    admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
    admin.list_topics(timeout=5)
    print("✅ Connected to Kafka")
except Exception as e:
    print(f"❌ Kafka failed: {e}")
    print("   Kafka may take 30-60s to start. Try again in a moment.")

## 📝 Step 1 — Understand the Data

Let's look at the ads we'll be tracking clicks for. The `ads` table was created automatically when the PostgreSQL container started (see `db/init.sql`).

In [ ]:
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT id, title, advertiser, budget FROM ads ORDER BY id")
rows = cur.fetchall()
conn.close()

print(f"{'ID':<4} {'Title':<42} {'Advertiser':<12} {'Budget':>10}")
print("-" * 72)
for row in rows:
    print(f"{row[0]:<4} {row[1]:<42} {row[2]:<12} ${row[3]:>9,.2f}")

print(f"\n📊 Total ads: {len(rows)}")

## 📤 Step 2 — Create a Kafka Topic

A **topic** is like a named channel in Kafka. We'll create a topic called `ad-clicks` where every message is one click event.

We'll give it **3 partitions**. Why? Because we want to partition by `ad_id`. All clicks for the *same ad* go to the *same partition*, which means:
- They arrive **in order** for that ad
- A single consumer can aggregate all clicks for one ad without coordination

Think of partitions like checkout lanes at a supermarket — more lanes means more throughput.

In [ ]:
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})

topic_name = "ad-clicks"
num_partitions = 3

# Check if the topic already exists
existing_topics = admin.list_topics(timeout=10).topics
if topic_name in existing_topics:
    print(f"ℹ️  Topic '{topic_name}' already exists with "
          f"{len(existing_topics[topic_name].partitions)} partition(s)")
else:
    # Create the topic with 3 partitions and replication factor 1 (single broker)
    new_topic = NewTopic(topic_name, num_partitions=num_partitions, replication_factor=1)
    futures = admin.create_topics([new_topic])
    for t, future in futures.items():
        future.result()  # block until done
        print(f"✅ Created topic '{t}' with {num_partitions} partitions")

# Show partition info
metadata = admin.list_topics(timeout=10)
topic_meta = metadata.topics[topic_name]
print(f"\n📦 Topic: {topic_name}")
for pid, pinfo in sorted(topic_meta.partitions.items()):
    print(f"   Partition {pid} — leader broker: {pinfo.leader}")

## 📤 Step 3 — Produce Click Events

Now let's simulate users clicking on ads. Each click event contains:

| Field | Description |
|-------|-------------|
| `ad_id` | Which ad was clicked |
| `impression_id` | Unique ID for this specific ad view (we'll use this for dedup in Notebook 3) |
| `user_id` | The user who clicked |
| `ip_address` | Their IP address |
| `user_agent` | Their browser info |
| `event_time` | When the click actually happened |

We **produce** (send) these events to Kafka. The **key** of each message is the `ad_id` — this tells Kafka which partition to put the message in.

In [ ]:
# Create a Kafka producer
producer = Producer({"bootstrap.servers": KAFKA_BROKER})

# A helper to generate a realistic click event
def make_click_event(ad_id: int) -> dict:
    """Generate a single simulated click event."""
    return {
        "ad_id": ad_id,
        "impression_id": str(uuid.uuid4()),
        "user_id": f"user_{random.randint(1, 500)}",
        "ip_address": f"192.168.{random.randint(1,254)}.{random.randint(1,254)}",
        "user_agent": random.choice([
            "Mozilla/5.0 (iPhone; CPU iPhone OS 17_0)",
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_0) Safari/17",
        ]),
        "event_time": datetime.now(timezone.utc).isoformat(),
        "run_id": RUN_ID,        # so the consumer below can ignore older runs
    }

# Callback that fires when Kafka confirms it received our message
delivery_reports = []

def on_delivery(err, msg):
    if err:
        print(f"❌ Delivery failed: {err}")
    else:
        delivery_reports.append({
            "topic": msg.topic(),
            "partition": msg.partition(),
            "offset": msg.offset()
        })

# Produce 20 click events for random ads
num_events = 20
for i in range(num_events):
    ad_id = random.randint(1, 10)  # we have 10 ads
    event = make_click_event(ad_id)

    # Key = ad_id (as string). Kafka hashes this to choose the partition.
    producer.produce(
        topic=topic_name,
        key=str(ad_id),
        value=json.dumps(event),
        callback=on_delivery
    )

# Flush waits until all messages are delivered
producer.flush(timeout=10)

print(f"✅ Produced {num_events} click events to topic '{topic_name}'")
print(f"\n📊 Delivery breakdown by partition:")
from collections import Counter
partition_counts = Counter(r["partition"] for r in delivery_reports)
for p in sorted(partition_counts):
    print(f"   Partition {p}: {partition_counts[p]} messages")

print("\n💡 Notice how messages are spread across partitions based on ad_id.")
print("   All clicks for the SAME ad go to the SAME partition.")

### 🔍 Go look in Kafka UI!

Open http://localhost:8081 → Topics → `ad-clicks` → Messages.  
You should see the 20 click events we just produced. Notice how the key is the `ad_id`.

## 📥 Step 4 — Consume Click Events

Now let's build a **consumer** that reads click events from Kafka. In a real system, this consumer would run forever in a loop. Here, we'll read the events we just produced and then stop.

### Key Concepts

- **Consumer Group**: Kafka tracks which messages each consumer group has already read. If the consumer crashes and restarts, it picks up where it left off.
- **Offset**: Each message in a partition has a sequence number (offset). The consumer commits its offset to tell Kafka "I've processed up to here."

In [ ]:
# Create a consumer that reads from the beginning of the topic
consumer = Consumer({
    "bootstrap.servers": KAFKA_BROKER,
    "group.id": "notebook-consumer-" + str(uuid.uuid4())[:8],  # unique group each run
    "auto.offset.reset": "earliest",  # start from the very first message
    "enable.auto.commit": False        # we'll commit manually after processing
})

consumer.subscribe([topic_name])

consumed_events = []
print("📥 Consuming click events from Kafka...\n")

# Poll for messages (try for up to 20 seconds)
start = time.time()
while time.time() - start < 20:
    msg = consumer.poll(timeout=1.0)
    if msg is None:
        continue
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            continue
        print(f"❌ Error: {msg.error()}")
        break

    event = json.loads(msg.value().decode("utf-8"))
    if event.get("run_id") != RUN_ID:
        continue  # left over from an earlier run of this notebook — skip it
    consumed_events.append(event)

    if len(consumed_events) <= 5:  # print first 5
        print(f"  Partition {msg.partition()} | Offset {msg.offset()} | "
              f"ad_id={event['ad_id']} user={event['user_id']}")

    if len(consumed_events) >= num_events:
        break

consumer.close()

if len(consumed_events) > 5:
    print(f"  ... and {len(consumed_events) - 5} more")

print(f"\n✅ Consumed {len(consumed_events)} events total")

# The topic is not empty when you re-run this notebook. If this assertion ever
# fires, the consumer read someone else's events and every count below is about
# the wrong data.
assert len(consumed_events) == num_events, (
    f"expected to read back the {num_events} events produced by run {RUN_ID}, "
    f"got {len(consumed_events)}"
)

## 💾 Step 5 — Store Events in PostgreSQL

Now let's write these events to the `click_events` table. In a production system, the consumer would do this in its processing loop. Here we'll insert the events we just consumed.

In [ ]:
conn = get_db()
cur = conn.cursor()

insert_sql = """
    INSERT INTO click_events (ad_id, impression_id, user_id, ip_address, user_agent, event_time)
    VALUES (%s, %s, %s, %s, %s, %s)
"""

inserted = 0
for event in consumed_events:
    cur.execute(insert_sql, (
        event["ad_id"],
        event["impression_id"],
        event["user_id"],
        event["ip_address"],
        event["user_agent"],
        event["event_time"]
    ))
    inserted += 1

conn.commit()
conn.close()

print(f"✅ Inserted {inserted} click events into PostgreSQL")

### 🔍 Go look in Adminer!

Open http://localhost:8080 → login → click on `click_events` table → Select data.  
You should see the rows we just inserted.

## 📊 Step 6 — Query Raw Events

Let's see what an advertiser might query. Below is the **naive approach** — counting clicks directly from the raw events table. This works at small scale, but becomes slow at 100M+ clicks per day.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Advertiser query: "How many clicks did each of my ads get?"
cur.execute("""
    SELECT a.title, a.advertiser, COUNT(ce.id) AS total_clicks,
           COUNT(DISTINCT ce.user_id) AS unique_users
    FROM ads a
    LEFT JOIN click_events ce ON a.id = ce.ad_id
    GROUP BY a.id, a.title, a.advertiser
    ORDER BY total_clicks DESC
""")

rows = cur.fetchall()
conn.close()

print(f"{'Ad Title':<42} {'Advertiser':<12} {'Clicks':>7} {'Unique':>7}")
print("-" * 72)
for row in rows:
    print(f"{row[0]:<42} {row[1]:<12} {row[2]:>7} {row[3]:>7}")

print("\n⚠️  This GROUP BY works fine with 20 events.")
print("   At 100M events/day, this query would take MINUTES.")
print("   That's why we need PRE-AGGREGATION (see Notebook 2).")

## 🏋️ Step 7 — Simulate Higher Throughput

Let's produce a larger batch to see how Kafka handles throughput. We'll send 1 000 events and measure how fast Kafka can ingest them.

In [ ]:
producer = Producer({
    "bootstrap.servers": KAFKA_BROKER,
    # Batching settings — Kafka groups messages together for efficiency
    "batch.size": 65536,      # batch up to 64 KB before sending
    "linger.ms": 10,          # wait up to 10ms to fill the batch
})

num_bulk = 1000
start = time.time()

for _ in range(num_bulk):
    ad_id = random.randint(1, 10)
    event = make_click_event(ad_id)
    producer.produce(
        topic=topic_name,
        key=str(ad_id),
        value=json.dumps(event)
    )

producer.flush(timeout=30)
elapsed = time.time() - start

print(f"✅ Produced {num_bulk} events in {elapsed:.2f} seconds")
print(f"   Throughput: {num_bulk / elapsed:,.0f} events/second")
print()
print("💡 Kafka can easily handle 100k+ messages/second per broker.")
print("   Our 10k clicks/second requirement is well within its capacity.")
print("   The bottleneck is usually the DATABASE, not Kafka — that's why")
print("   we buffer through Kafka first.")

## 🔥 Step 8 — Hot Shard Mitigation

So far we partition by `ad_id`: all clicks for the same ad go to the same Kafka partition. That preserves order and makes aggregation easy — **but it has a dark side**.

### The Problem: One Viral Ad = One Overloaded Partition

Imagine the Super Bowl airs a Nike ad and suddenly **80% of all clicks** target `ad_id=1`. Because we partition by `ad_id`, every one of those clicks lands on the *same* Kafka partition and gets processed by the *same* consumer.

```
Partition 0:  ad_id=1  ████████████████████████  ← HOT SHARD
Partition 1:  ad_id=2  ██
Partition 2:  ad_id=3  █
```

Partition 0's consumer falls behind. The others sit idle. This is a **hot shard** (sometimes called a "hot key").

### The Fix: Split the Hot Key

If an ad is known to be hot, append a small **random suffix** so its clicks fan out across multiple partitions:

| Before (hot) | After (split) |
|---|---|
| key = `"1"` | key = `"1:0"`, `"1:1"`, `"1:2"`, ... |

Now the clicks for `ad_id=1` spread across many partitions, different consumers share the load, and we just SUM across the sub-keys at query time.

Trade-off: we lose strict per-ad ordering (rarely needed for *counting*) in exchange for much higher throughput.

And one honest limit: sub-sharding can only spread a key across the partitions that
*exist*. With 3 partitions the best possible outcome is 33 % each — splitting into
32 sub-keys gets close to that and no further. Past that point the fix is to add
partitions, not more suffixes.

Let's demonstrate by simulating a skewed click distribution and comparing both strategies.


In [ ]:
import collections
import zlib

NUM_CLICKS = 2000
NUM_PARTITIONS = 3
HOT_AD_ID = 1
HOT_AD_SHARE = 0.80    # 80% of traffic goes to the hot ad
HOT_SUFFIX_RANGE = 32  # split the hot ad into 32 logical sub-shards


def assign_partition(key: str, num_partitions: int) -> int:
    """Stand-in for Kafka's default partitioner.

    Kafka hashes the key BYTES (murmur2) and takes the remainder. We use crc32,
    which is also a stable hash of the bytes — the property that matters here is
    stability. Python's built-in hash() is salted per process (PYTHONHASHSEED),
    so using it would send every click to a different partition on every kernel
    restart and this whole comparison would be noise.
    """
    return zlib.crc32(key.encode("utf-8")) % num_partitions


# One click stream, keyed two different ways. Seeding it means the numbers below
# are the same on every machine, so the two strategies really are comparable.
random.seed(20)
click_stream = [
    HOT_AD_ID if random.random() < HOT_AD_SHARE else random.randint(2, 10)
    for _ in range(NUM_CLICKS)
]
random.seed(21)
suffixes = [random.randint(0, HOT_SUFFIX_RANGE - 1) for _ in range(NUM_CLICKS)]

# ── BAD: partition by ad_id alone ──
bad_dist = collections.Counter(
    assign_partition(str(ad_id), NUM_PARTITIONS) for ad_id in click_stream
)

# ── BETTER: append a random suffix for the hot ad ──
better_dist = collections.Counter(
    assign_partition(f"{ad_id}:{suffix}" if ad_id == HOT_AD_ID else str(ad_id),
                     NUM_PARTITIONS)
    for ad_id, suffix in zip(click_stream, suffixes)
)

bad_max_share = max(bad_dist.values()) / NUM_CLICKS
better_max_share = max(better_dist.values()) / NUM_CLICKS
even_share = 1 / NUM_PARTITIONS

print(f"🔥 Hot Shard Simulation ({NUM_CLICKS} clicks, ad_id={HOT_AD_ID} gets "
      f"{HOT_AD_SHARE:.0%})")
print("=" * 55)
print()
print("BAD  — partition by ad_id only:")
for p in range(NUM_PARTITIONS):
    bar = "█" * (bad_dist[p] // 30)
    print(f"  Partition {p}: {bad_dist[p]:>4} clicks  {bar}")
print(f"  busiest partition holds {bad_max_share:.0%} of all traffic")

print()
print(f"BETTER — random suffix (0-{HOT_SUFFIX_RANGE - 1}) for the hot ad:")
for p in range(NUM_PARTITIONS):
    bar = "█" * (better_dist[p] // 30)
    print(f"  Partition {p}: {better_dist[p]:>4} clicks  {bar}")
print(f"  busiest partition holds {better_max_share:.0%} of all traffic")

print()
print(f"💡 The BAD strategy crushes one partition ({bad_max_share:.0%}) while the")
print("   others sit idle. Splitting the hot key flattens that considerably.")
print(f"   But note the floor: perfectly even across {NUM_PARTITIONS} partitions is")
print(f"   {even_share:.0%} each, and no amount of sub-sharding can beat that.")
print("   Once you are near it, the next move is MORE PARTITIONS, not more sub-keys.")
print("   Trade-off: we lose strict per-ad ordering, but counting stays exact —")
print("   we just SUM across the sub-keys at query time.")

# ── Assertions: the lesson has to keep reproducing ──────────────────────────
assert sum(bad_dist.values()) == sum(better_dist.values()) == NUM_CLICKS, \
    "the simulation lost clicks"
assert bad_max_share > 0.70, \
    f"the hot shard should be genuinely hot; busiest partition held {bad_max_share:.0%}"
assert better_max_share < bad_max_share / 1.7, \
    ("splitting the hot key should visibly flatten the skew; got "
     f"{better_max_share:.0%} vs {bad_max_share:.0%}")
assert better_max_share >= even_share, \
    "no keying scheme can put less than 1/num_partitions on the busiest partition"


### 🧠 How Do You Detect a Hot Ad in Production?

You usually don't hard-code it. A typical approach:

1. A background job watches recent click counts per ad (say, the last 5 minutes).
2. Any ad exceeding a threshold (e.g., > 1 000 clicks/sec) gets added to a **hot-ad set** in Redis.
3. The click producer checks this set before choosing a partition key: if the ad is hot, it adds a random suffix; otherwise it uses `ad_id` directly.

This adaptive approach keeps ordering strong for normal ads and only "shards" the ads that actually need it.

**Real-world example**: TikTok uses this exact pattern for viral videos — their metrics pipeline watches for sudden spikes and dynamically re-partitions hot keys across many workers.


## 🧹 Cleanup

In [ ]:
# Clean up click_events we inserted (keep the table for Notebook 2)
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_events")
deleted = cur.rowcount
conn.commit()
conn.close()
print(f"🧹 Deleted {deleted} rows from click_events")
print("   (The Kafka topic still has messages — that's fine for Notebook 2)")

## 📚 Summary

### Key Takeaways

1. **Kafka is a buffer** — it absorbs bursts of writes so the database doesn't get overwhelmed
2. **Partition by ad_id** — all clicks for the same ad land in the same partition, preserving order
3. **...until one ad goes viral** — that same key becomes a hot shard. Trade ordering for
   throughput by appending a random suffix, but remember the ceiling: you can never spread
   a key across more partitions than the topic has
4. **Producers are fast** — Kafka can ingest thousands of messages per second with batching
5. **Consumers read at their own pace** — if the consumer falls behind, messages wait safely in Kafka
6. **Raw event queries don't scale** — GROUP BY on millions of rows is too slow for advertisers

### What's Next

In **Notebook 2**, we'll build a **real-time aggregation consumer** that reads events from Kafka and maintains pre-aggregated click counts in 1-minute windows — so advertiser queries return instantly.